# 벤치마크 데이터셋 실습

**Benchmark · Matbench · 표준 평가셋**

같은 데이터와 같은 분할로 모델을 비교하도록 정해둔 평가용 데이터셋.

소재 분야에서 이해하기: 자체 모델을 공개 분할 기준으로 기존 결과와 비교한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 같은 분할로 비교해야 합니다

분할이 다르면 모델 비교 결과가 뒤집힐 수 있습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = [('Ridge', make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
              ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=0)),
              ('GradientBoosting', GradientBoostingRegressor(random_state=0))]

print('분할 시드마다 순위가 바뀌는지 확인:')
for seed in range(5):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores = [(mean_absolute_error(y_te, model.fit(X_tr, y_tr).predict(X_te)), name)
              for name, model in candidates]
    scores.sort()
    print('  seed %d 최고: %-17s (%.2f)  |  ' % (seed, scores[0][1], scores[0][0])
          + ', '.join('%s %.2f' % (name, value) for value, name in scores))

## 2. 고정된 분할과 반복 평가

벤치마크는 분할을 고정해 공개하고, 여러 폴드의 평균과 산포를 함께 보고합니다.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(5, shuffle=True, random_state=42)      # 벤치마크가 공개한 고정 분할
print('고정 분할 기준 결과:')
for name, model in candidates:
    scores = -cross_val_score(model, X, y, cv=folds, scoring='neg_mean_absolute_error')
    print('  %-17s MAE %.2f ± %.2f' % (name, scores.mean(), scores.std()))
print('\n산포보다 작은 차이는 "더 좋다"고 말하기 어렵습니다.')
print('Matbench 처럼 분할과 지표가 정해진 벤치마크를 쓰면 논문 사이 비교가 가능해집니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#benchmark-dataset)을 여세요.